In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install pyttsx3

In [ ]:
import sys
!{sys.executable} -m pip install mediapipe==0.10.11

In [ ]:
import sys
!{sys.executable} -m pip install gTTS pygame

In [ ]:
pip install opencv-python mediapipe numpy gTTS pygame

In [ ]:
import sys
!{sys.executable} -m pip install ultralytics

In [3]:
import cv2
import mediapipe as mp
import math
import numpy as np
import threading
import time
import winsound
from gtts import gTTS
import pygame
import io
from ultralytics import YOLO

print("=" * 70)
print("     🇪🇬 EGYPT ADAS - ULTIMATE MASTER SYSTEM (DISTRACTION FIXED)")
print("     Initializing YOLO, Face Mesh & Hands...")
print("=" * 70)


yolo_model = YOLO("yolov8n.pt")

pygame.mixer.init()

mp_drawing = mp.solutions.drawing_utils
mp_face_mesh = mp.solutions.face_mesh
mp_hands = mp.solutions.hands

LEFT_EYE = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
MOUTH = [78, 308, 13, 14]

def euclidean_distance(p1, p2):
    return math.hypot(p2[0] - p1[0], p2[1] - p1[1])

def calculate_normalized_ear(eye_points, landmarks, img_w, img_h):
    pts = [(landmarks.landmark[i].x * img_w, landmarks.landmark[i].y * img_h) for i in eye_points]
    v1 = euclidean_distance(pts[1], pts[5])
    v2 = euclidean_distance(pts[2], pts[4])
    h = euclidean_distance(pts[0], pts[3])
    
    face_left = (landmarks.landmark[234].x * img_w, landmarks.landmark[234].y * img_h)
    face_right = (landmarks.landmark[454].x * img_w, landmarks.landmark[454].y * img_h)
    face_width = euclidean_distance(face_left, face_right)
    return (v1 + v2) / (2.0 * h) if h != 0 else 0

def calculate_normalized_mar(landmarks, img_w, img_h):
    pts = [(landmarks.landmark[i].x * img_w, landmarks.landmark[i].y * img_h) for i in MOUTH]
    v = euclidean_distance(pts[2], pts[3])
    h = euclidean_distance(pts[0], pts[1])
    return v / h if h != 0 else 0


program_running = True
current_buzzer_level = 0
is_speaking = False

last_voice_times = {
    "FAINTING": 0,
    "PHONE": 0,
    "DROWSY": 0,
    "STRESS": 0,
    "DISTRACTION": 0,
    "YAWN": 0
}

VOICE_COOLDOWN = {
    "FAINTING": 10,
    "PHONE": 8,
    "DROWSY": 8,
    "STRESS": 8,
    "DISTRACTION": 4,
    "YAWN": 8
}

def buzzer_worker():
    global current_buzzer_level, program_running
    while program_running:
        if current_buzzer_level > 0:
            if current_buzzer_level == 3:
                winsound.Beep(2800, 300)
                time.sleep(0.05)
            else: # تحذير عادي
                winsound.Beep(2000, 300)
                time.sleep(0.15)
        else:
            time.sleep(0.1)

threading.Thread(target=buzzer_worker, daemon=True).start()

def speak_alert_arabic(text, alert_key):
    global is_speaking
    current_time = time.time()
    if current_time - last_voice_times.get(alert_key, 0) < VOICE_COOLDOWN.get(alert_key, 5):
        return
    
    if is_speaking:
        return
        
    last_voice_times[alert_key] = current_time
    is_speaking = True
    
    def run_tts():
        global is_speaking
        try:
            tts = gTTS(text=text, lang='ar', slow=False)
            fp = io.BytesIO()
            tts.write_to_fp(fp)
            fp.seek(0)
            pygame.mixer.music.load(fp)
            pygame.mixer.music.play()
            while pygame.mixer.music.get_busy():
                time.sleep(0.1)
        except Exception as e:
            pass
        is_speaking = False
    threading.Thread(target=run_tts, daemon=True).start()

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# متغيرات المعايرة
is_calibrated = False
calib_start_time = time.time()
calib_ears = []
calib_mars = []
BASE_EAR = 0.25
BASE_MAR = 0.20

# متغيرات الزمن
eye_closed_start_time = None
yawn_start_time = None
phone_interaction_start_time = None
distraction_start_time = None
stress_start_time = None

smoothed_yaw = 0.0
alpha = 0.3

with mp_face_mesh.FaceMesh(max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5, min_tracking_confidence=0.5) as face_mesh, \
     mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.5, min_tracking_confidence=0.5) as hands:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break

        image = cv2.flip(image, 1)
        h, w, c = image.shape
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        current_time = time.time()
        
        status_text = "SAFE & FOCUSED"
        status_color = (0, 255, 0)
        current_buzzer_level = 0
        

        dbg_ear, dbg_mar, dbg_yaw = 0.0, 0.0, 0.0
        dbg_phone, dbg_hand, dbg_stress = "No", "No", "No"

        yolo_results = yolo_model(image, verbose=False)
        phone_detected_by_yolo = False
        phone_boxes = []
        for r in yolo_results:
            for box in r.boxes:
                if int(box.cls[0]) == 67 and float(box.conf[0]) > 0.40:
                    phone_detected_by_yolo = True
                    x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                    phone_boxes.append((x1, y1, x2, y2))
                    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)
                    cv2.putText(image, "CELL PHONE", (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

        dbg_phone = "Yes" if phone_detected_by_yolo else "No"


        hand_results = hands.process(image_rgb)
        hand_using_phone = False
        hand_on_head = False
        hand_centers = []
        
        if hand_results.multi_hand_landmarks:
            for hand_landmarks in hand_results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                hx = int(hand_landmarks.landmark[9].x * w)
                hy = int(hand_landmarks.landmark[9].y * h)
                hand_centers.append((hx, hy))

        for (px1, py1, px2, py2) in phone_boxes:
            for (hx, hy) in hand_centers:
                if px1 - 50 <= hx <= px2 + 50 and py1 - 50 <= hy <= py2 + 50:
                    hand_using_phone = True
                    break

        dbg_hand = "Yes" if hand_using_phone else "No"


        face_results = face_mesh.process(image_rgb)
        face_landmarks_list = face_results.multi_face_landmarks
        
        is_fainting = False
        is_drowsy = False
        is_phone_use = False
        is_distracted = False
        is_stressed = False
        is_yawning = False

        if face_landmarks_list:
            for face_landmarks in face_landmarks_list:
                
                mp_drawing.draw_landmarks(image=image, landmark_list=face_landmarks, connections=mp_face_mesh.FACEMESH_TESSELATION,
                    landmark_drawing_spec=None, connection_drawing_spec=mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=1, circle_radius=1))
                
                forehead_y = face_landmarks.landmark[10].y * h
                nose_y = face_landmarks.landmark[1].y * h
                
                if hand_results.multi_hand_landmarks:
                    for hand_landmarks in hand_results.multi_hand_landmarks:
                        wrist_y = hand_landmarks.landmark[0].y * h
                        if wrist_y < nose_y:
                            hand_on_head = True

                dbg_stress = "Yes" if hand_on_head else "No"

   
                face_3d, face_2d = [], []
                for idx in [33, 263, 1, 61, 291, 199]:
                    lm = face_landmarks.landmark[idx]
                    x, y = int(lm.x * w), int(lm.y * h)
                    face_2d.append([x, y])
                    face_3d.append([x, y, lm.z])
                
                face_2d = np.array(face_2d, dtype=np.float64)
                face_3d = np.array(face_3d, dtype=np.float64)
                cam_matrix = np.array([[w, 0, w / 2], [0, w, h / 2], [0, 0, 1]], dtype=np.float64)
                dist_matrix = np.zeros((4, 1), dtype=np.float64)
                _, rot_vec, _ = cv2.solvePnP(face_3d, face_2d, cam_matrix, dist_matrix)
                rmat, _ = cv2.Rodrigues(rot_vec)
                angles, _, _, _, _, _ = cv2.RQDecomp3x3(rmat)

                raw_yaw = angles[1] * 360
                smoothed_yaw = (alpha * raw_yaw) + ((1 - alpha) * smoothed_yaw)
                dbg_yaw = smoothed_yaw

                left_ear = calculate_normalized_ear(LEFT_EYE, face_landmarks, w, h)
                right_ear = calculate_normalized_ear(RIGHT_EYE, face_landmarks, w, h)
                avg_ear = (left_ear + right_ear) / 2.0
                mar = calculate_normalized_mar(face_landmarks, w, h)
                
                dbg_ear = avg_ear
                dbg_mar = mar

         
                if not is_calibrated:
                    calib_ears.append(avg_ear)
                    calib_mars.append(mar)
                    cv2.putText(image, "CALIBRATING... LOOK FORWARD", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
                    if current_time - calib_start_time > 4.0:
                        BASE_EAR = np.median(calib_ears)
                        BASE_MAR = np.median(calib_mars)
                        is_calibrated = True
                    cv2.imshow('Egypt ADAS - Master System', image)
                    if cv2.waitKey(1) & 0xFF == ord('q'):
                        break
                    continue

                # 1. النعاس والإغماء
                ear_threshold = BASE_EAR * 0.58
                if avg_ear < ear_threshold:
                    if eye_closed_start_time is None:
                        eye_closed_start_time = current_time
                    closed_duration = current_time - eye_closed_start_time
                    
                    if closed_duration >= 3.0:
                        is_fainting = True
                    elif closed_duration >= 1.0:
                        is_drowsy = True
                else:
                    eye_closed_start_time = None

       
                if abs(smoothed_yaw) > 10:
                    if distraction_start_time is None:
                        distraction_start_time = current_time
                    elif current_time - distraction_start_time >= 0.5:
                        is_distracted = True
                else:
                    distraction_start_time = None


                if phone_detected_by_yolo and hand_using_phone:
                    if phone_interaction_start_time is None:
                        phone_interaction_start_time = current_time
                    elif current_time - phone_interaction_start_time >= 1.0:
                        is_phone_use = True
                else:
                    phone_interaction_start_time = None

            
                if hand_on_head:
                    if stress_start_time is None:
                        stress_start_time = current_time
                    elif current_time - stress_start_time >= 0.8:
                        is_stressed = True
                else:
                    stress_start_time = None

    
                if mar > (BASE_MAR * 2.2):
                    if yawn_start_time is None:
                        yawn_start_time = current_time
                    elif current_time - yawn_start_time >= 1.2:
                        is_yawning = True
                else:
                    yawn_start_time = None


        else:

            if distraction_start_time is None:
                distraction_start_time = current_time
            elif current_time - distraction_start_time >= 0.5:
                is_distracted = True

 
        if is_fainting:
            status_text = "SOS: FAINTING / EMERGENCY!"
            status_color = (0, 0, 255)
            current_buzzer_level = 3
            speak_alert_arabic("تنبيه طوارئ، السائق فاقد للوعي!", "FAINTING")
        elif is_phone_use:
            status_text = "DISTRACTED: PHONE USAGE!"
            status_color = (0, 0, 255)
            current_buzzer_level = 2
            speak_alert_arabic("انتبه، استخدام الهاتف ممنوع أثناء القيادة!", "PHONE")
        elif is_drowsy:
            status_text = "DROWSY - SLEEPING!"
            status_color = (0, 0, 255)
            current_buzzer_level = 2
            speak_alert_arabic("انتبه، استيقظ وركز في الطريق!", "DROWSY")
        elif is_stressed:
            status_text = "STRESS / ANGER DETECTED!"
            status_color = (0, 0, 255)
            current_buzzer_level = 2
            speak_alert_arabic("يرجى الهدوء والحفاظ على أعصابك أثناء القيادة", "STRESS")
        elif is_distracted:
            status_text = "DISTRACTED: LOOKING AWAY!"
            status_color = (0, 165, 255)
            current_buzzer_level = 1
            speak_alert_arabic("انتبه للطريق، لا تتلفت كثيراً", "DISTRACTION")
        elif is_yawning:
            status_text = "FATIGUE - YAWNING!"
            status_color = (0, 165, 255)
            current_buzzer_level = 1
            speak_alert_arabic("يبدو أنك مرهق، أخذ قسط من الراحة", "YAWN")

        # واجهة الـ UI
        cv2.rectangle(image, (10, 10), (640, 125), (20, 20, 20), -1)
        cv2.putText(image, f"STATUS: {status_text}", (20, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
        
        debug_str = f"EAR:{dbg_ear:.2f} | MAR:{dbg_mar:.2f} | Yaw:{dbg_yaw:.1f}"
        cv2.putText(image, debug_str, (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 1)
        
        debug_str2 = f"Phone:{dbg_phone} | Hand Near Phone:{dbg_hand} | Stress:{dbg_stress}"
        cv2.putText(image, debug_str2, (20, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

        cv2.imshow('Egypt ADAS - Ultimate Master System', image)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

program_running = False
cap.release()
cv2.destroyAllWindows()
print("Ultimate Master ADAS System closed successfully.")

     🇪🇬 EGYPT ADAS - ULTIMATE MASTER SYSTEM (DISTRACTION FIXED)
     Initializing YOLO, Face Mesh & Hands...
Ultimate Master ADAS System closed successfully.
